In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 7.3 Transforming Higher Ed Text Data to Vectors
- CountVectorizer vs TF-IDF
- Ordinal items are already numeric, text needs vectorization
- Merging text vectors with structured survey features

## Setup

In [ ]:
import pandas as pd
import numpy as np

pd.options.display.max_columns = None

ML_Survey_Data = pd.read_csv('../data/ML_Survey_Data.csv')
ML_Survey_Data22 = pd.read_csv('../data/ML_Survey_Data22.csv')
display(ML_Survey_Data)

## CountVectorizer vs TF-IDF
- **CountVectorizer** — raw word counts (Bag-of-Words). Simple, ignores word order.
- **TF-IDF** — weights words by how distinctive they are across the corpus. Usually the better default.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(stop_words='english', lowercase=True, ngram_range=(1, 1))
count_matrix = count_vectorizer.fit_transform(ML_Survey_Data['Free_Response_Text'])
count_feature_names = count_vectorizer.get_feature_names_out()

df_count_vectorized = pd.DataFrame(count_matrix.toarray(), columns=count_feature_names)
df_count_vectorized.index = ML_Survey_Data.index
print(f"CountVectorized shape: {df_count_vectorized.shape}")
df_count_vectorized

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(stop_words='english', lowercase=True, ngram_range=(1, 1))
tfidf_matrix = tfidf_vectorizer.fit_transform(ML_Survey_Data['Free_Response_Text'])
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

df_tfidf_vectorized = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_feature_names)
df_tfidf_vectorized.index = ML_Survey_Data.index
print(f"TfidfVectorized shape: {df_tfidf_vectorized.shape}")
df_tfidf_vectorized

## Reusable Helper Function
Wraps both vectorizers so we can apply the same steps to train and test consistently.

In [ ]:
def vectorize_text_data(dataframe):
    count_vectorizer = CountVectorizer(stop_words='english', lowercase=True, ngram_range=(1, 1))
    count_matrix = count_vectorizer.fit_transform(dataframe['Free_Response_Text'])
    df_count_vectorized = pd.DataFrame(count_matrix.toarray(), columns=count_vectorizer.get_feature_names_out())
    df_count_vectorized.index = dataframe.index

    tfidf_vectorizer = TfidfVectorizer(stop_words='english', lowercase=True, ngram_range=(1, 1))
    tfidf_matrix = tfidf_vectorizer.fit_transform(dataframe['Free_Response_Text'])
    df_tfidf_vectorized = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
    df_tfidf_vectorized.index = dataframe.index

    return df_count_vectorized, df_tfidf_vectorized

df_count_vectorized, df_tfidf_vectorized = vectorize_text_data(ML_Survey_Data)
df_count_vectorized22, df_tfidf_vectorized22 = vectorize_text_data(ML_Survey_Data22)
print("Vectorized train and test.")

## Merge Text Vectors with Structured Survey Features
Keep the 10 NSSE ordinal columns, append the TF-IDF word-weight columns.

In [ ]:
df_merged_surv_features = pd.merge(ML_Survey_Data.iloc[:, :11], df_tfidf_vectorized, left_index=True, right_index=True)
df_merged_surv_features22 = pd.merge(ML_Survey_Data22.iloc[:, :11], df_tfidf_vectorized22, left_index=True, right_index=True)

print(f"Merged train shape: {df_merged_surv_features.shape}")
print(f"Merged test shape:  {df_merged_surv_features22.shape}")
display(df_merged_surv_features.head())

## Export

In [ ]:
import os
os.makedirs('../data/', exist_ok=True)

df_merged_surv_features.to_csv('../data/ML_Survey_Data_Num.csv', index=False)
df_merged_surv_features22.to_csv('../data/ML_Survey_Data22_Num.csv', index=False)
print("Saved ML_Survey_Data_Num.csv and ML_Survey_Data22_Num.csv to ../data/")

## Summary
- CountVectorizer → raw frequencies. TF-IDF → weighted by distinctiveness, usually the better default.
- Both ignore word order and semantic meaning — "not helpful" and "very helpful" look similar to a bag-of-words model.

**Next:** 7.4 uses these same vectors for topic modeling.